# TFM RAG Evaluation — GPT-5.4-mini (OpenAI API)

Evalúa las 4 variantes A/B/C/D del pipeline RAG usando **gpt-5.4-mini** como LLM.
No necesita GPU para el LLM (llamadas a API), pero sí para BGE-M3 y el reranker.

## Orden de ejecución
1. Cell 1 — Check GPU
2. Cell 2 — Clone repo + instalar dependencias
3. Cell 3 — Crear `.env` con OpenAI API
4. Cell 4 — Ingestar documentos en Qdrant
5. Cell 5 — Evaluar A/B/C/D con gpt-5.4-mini (blocking, ~50-70 min)


In [ ]:
# Cell 1 — Check GPU (opcional, no bloquea)
import subprocess
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip() or 'No GPU detected')
except FileNotFoundError:
    print('nvidia-smi no encontrado — sesión CPU o GPU no activada')
    print('Puedes continuar: BGE-M3 y reranker funcionan en CPU (más lento).')
print('(GPU útil para BGE-M3 embeddings y reranker, no para el LLM — OpenAI API)')


In [ ]:
# Cell 2 — Clone repo + instalar dependencias
import subprocess, sys, os, getpass

gh_token = getpass.getpass('GitHub token: ')
subprocess.run(f'git clone https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git /content/TFM',
               shell=True, check=True)
subprocess.run(f'git -C /content/TFM remote set-url origin https://{gh_token}@github.com/nuriaiglesias/private-assistant-rag.git', shell=True)
subprocess.run('git -C /content/TFM config user.email "nuriait2001@gmail.com"', shell=True)
subprocess.run('git -C /content/TFM config user.name "NuriaIglesias"', shell=True)
print('✅ Repo clonado')

os.chdir('/content/TFM')
os.makedirs('logs', exist_ok=True)

print('Instalando dependencias...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', 'assistant/requirements.txt'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
else:
    print('✅ Dependencias instaladas')


In [ ]:
# Cell 3 — Crear .env con OpenAI API (gpt-5.4-mini)
import getpass, os

openai_api_key = getpass.getpass('OpenAI API key (sk-...): ')

env_content = f"""LLM_PROVIDER=vllm
LLM_BASE_URL=https://api.openai.com
LLM_API_KEY={openai_api_key}
LLM_MODEL=gpt-5.4-mini
LLM_TEMPERATURE=1.0
LLM_MAX_TOKENS=512
LLM_TIMEOUT_SECONDS=120
RERANKER_MODEL=BAAI/bge-reranker-v2-m3
RAG_CANDIDATE_K=20
RAG_MIN_SCORE=0.0
QDRANT_URL=/content/qdrant_db
HYBRID_CORPUS_PATH=assistant/corpus/processed_md
HYBRID_DENSE_WEIGHT=0.5
HYBRID_BM25_WEIGHT=0.5
PHOENIX_ENABLED=false
PHOENIX_HOST=localhost
PHOENIX_PORT=4317
PHOENIX_PROJECT=assistant-rag
"""

with open('/content/TFM/assistant/.env', 'w') as f:
    f.write(env_content)
print('✅ .env creado (modelo=gpt-5.4-mini, temperature=1.0)')
print('Nota: gpt-5.4-mini tiene capacidad de razonamiento; temperature=1.0 es el valor recomendado.')


In [ ]:
# Cell 4 — Ingestar documentos en Qdrant (~5-10 min)
import subprocess, sys, os

os.chdir('/content/TFM')

ENV = {
    **os.environ,
    'PYTHONPATH': '/content/TFM/assistant/src',
    'QDRANT_URL': '/content/qdrant_db',
    'TRANSFORMERS_OFFLINE': '0',
    'HF_HUB_OFFLINE': '0',
}

print('Ingiriendo documentos en Qdrant (descarga BGE-M3 si es primera vez)...')
r = subprocess.run(
    [sys.executable, 'assistant/scripts/ingest.py'],
    env=ENV, capture_output=True, text=True, timeout=1800
)
print(r.stdout[-2000:] if r.stdout else '')
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    print('❌ Ingesta fallida')
else:
    print('✅ Ingesta completada')


In [ ]:
# Cell 5 — Evaluar A/B/C/D con gpt-5.4-mini (blocking, ~50-60 min)
# Incluye BERTScore. Resultados guardados en GitHub automáticamente.
import subprocess, sys, os, datetime

os.chdir('/content/TFM')

ENV = {
    **os.environ,
    'PYTHONPATH': '/content/TFM/assistant/src',
    'QDRANT_URL': '/content/qdrant_db',
    'LLM_PROVIDER': 'vllm',
    'LLM_BASE_URL': 'https://api.openai.com',
    'LLM_MODEL': 'gpt-5.4-mini',
    'LLM_TIMEOUT_SECONDS': '120',
    'PHOENIX_ENABLED': 'false',
}

# Leer API key del .env
with open('/content/TFM/assistant/.env') as f:
    for line in f:
        if line.startswith('LLM_API_KEY='):
            ENV['LLM_API_KEY'] = line.split('=', 1)[1].strip()

LOG = 'logs/openai_eval.log'

def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')

def save_results(label):
    subprocess.run('git -C /content/TFM add assistant/results/ logs/', shell=True)
    rc = subprocess.run(
        f'git -C /content/TFM commit -m "eval-openai: {label} gpt-5.4-mini results"',
        shell=True, capture_output=True, text=True
    )
    if 'nothing to commit' in rc.stdout:
        log('  (sin cambios nuevos)')
        return
    rp = subprocess.run('git -C /content/TFM push origin main',
                        shell=True, capture_output=True, text=True)
    log('  ✅ Guardado en GitHub' if rp.returncode == 0 else f'  ⚠️ Push fallido: {rp.stderr[:150]}')

def run(label, cmd):
    log(f'START — {label}')
    r = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    with open(LOG, 'a') as f:
        f.write((r.stdout + r.stderr)[-3000:] + '\n')
    if r.returncode == 0:
        log(f'DONE  — {label}')
    else:
        log(f'ERROR — {label} (exit {r.returncode})')
    save_results(label)

log('=' * 55)
log('Evaluación con gpt-5.4-mini — dataset privado UNIR')
log('=' * 55)

py = sys.executable
base = [py, 'assistant/scripts/eval/evaluate_answers.py']

run('A: Dense',              base)
run('B: Hybrid',             base + ['--use-hybrid'])
run('C: Hybrid+Reranking',   base + ['--use-hybrid', '--use-reranking'])
run('D: Orchestrator',       base + ['--use-hybrid', '--use-reranking', '--use-orchestrator'])
run('BERTScore',             [py, 'assistant/scripts/eval/compute_bertscore.py'])

log('=' * 55)
log('TODAS LAS VARIANTES COMPLETADAS')
log('Resultados en assistant/results/')
log('=' * 55)


In [ ]:
# Cell 6 — Ver resultados resumidos
import json, glob

files = sorted(glob.glob('/content/TFM/assistant/results/answers_results_*.jsonl'), reverse=True)[:4]

print(f'{'Variante':<30} {'n':>4} {'Fact-Cov':>10} {'ROUGE-L':>9} {'BERTScore':>10}')
print('-' * 70)

for fname in reversed(files):
    lines = [json.loads(l) for l in open(fname) if l.strip()]
    if not lines: continue
    first = lines[0]
    h = first.get('use_hybrid', False)
    r = first.get('use_reranking', False)
    o = first.get('use_orchestrator', False)
    label = {(False,False,False): 'A  Dense',
             (True,False,False):  'B  Hybrid',
             (True,True,False):   'C  Hybrid+Rerank',
             (True,True,True):    'D  Orchestrator'}.get((h,r,o), '?')
    ans = [l for l in lines if l.get('answerable', True)]
    fc  = sum(l.get('score',0) for l in ans) / len(ans) if ans else 0
    rl  = sum(l.get('rouge_l',0) for l in ans if l.get('rouge_l') is not None) / len(ans) if ans else 0
    bt_vals = [l.get('bertscore_f1') for l in ans if l.get('bertscore_f1') is not None]
    bt  = sum(bt_vals)/len(bt_vals) if bt_vals else None
    bt_str = f'{bt:.3f}' if bt else 'pending'
    print(f'{label:<30} {len(lines):>4} {fc:>10.3f} {rl:>9.3f} {bt_str:>10}')
